# 01 - Construccion del dataset

Proyecto 3, Text-to-SQL con LoRA. Este notebook levanta MySQL 8.4.10, carga los 12
esquemas propios, los puebla con datos reproducibles y genera los pares pregunta-SQL.

La validacion por ejecucion y el reporte de descartes estan en `scripts/validar_pares.py`.

## 1. Entorno

En Colab se instala el binario oficial de MySQL 8.4.10. En local basta con Docker:
`bash scripts/mysql_setup.sh docker`.

In [ ]:
!pip install -q PyMySQL
!bash scripts/mysql_setup.sh colab

In [ ]:
import os

os.environ['MYSQL_HOST'] = '127.0.0.1'
os.environ['MYSQL_PORT'] = '3306'
os.environ['MYSQL_USER'] = 'root'
os.environ['MYSQL_PASSWORD'] = 'proyecto3'

## 2. Carga de los esquemas

Una base por archivo `.sql`. El poblador es generico: descubre columnas y claves
foraneas en `information_schema` y genera valores con semilla 2026, de forma que
las mismas filas se reproducen en cualquier maquina.

In [ ]:
!python scripts/cargar_esquemas.py --dir datos/esquemas --poblar

## 3. El esquema tal como lo vera el modelo

El evaluador entrega a `generar_sql` las sentencias `SHOW CREATE TABLE`, con sus
`COMMENT`. Este es el formato exacto que usamos para entrenar.

In [ ]:
import sys

sys.path.insert(0, 'scripts')
import db

conn = db.conectar()
print(db.show_create(conn, 'biblioteca')[:1200])

## 4. Generacion de los pares

32 plantillas parametricas repartidas en los tres niveles de la evaluacion oculta.
Cada plantilla se instancia con columnas reales y con valores muestreados de la
propia base, para que los filtros no devuelvan conjuntos vacios.

In [ ]:
!python scripts/generar_pares.py --limite 12

In [ ]:
import json

import pandas as pd

pares = [json.loads(l) for l in open('datos/pares_crudos.jsonl', encoding='utf-8')]
df = pd.DataFrame(pares)
print(len(df), 'pares generados')
display(df.groupby(['nivel']).size().rename('pares'))
display(df.groupby(['base']).size().rename('pares'))

In [ ]:
for _, fila in df.sample(6, random_state=2026).iterrows():
    print('[%s/%s] %s' % (fila['base'], fila['nivel'], fila['pregunta']))
    print('   ', fila['sql'], '\n')

## 5. Siguiente paso

`scripts/validar_pares.py` ejecuta cada consulta en MySQL 8.4.10, descarta las que
fallan o devuelven un resultado vacio, adjunta el esquema de su base y escribe
`datos/entrenamiento.jsonl` junto con `datos/descartes.csv`.